# HDS-M03 Mini Project: Lung Segmentation from Chest X-rays

## Part 2: Thresholding (steps 4–5)

A threshold $t$ splits the pixels into two groups. Since lungs are dark, lung $=$ pixels with
intensity $< t$. The question in this notebook is how to choose $t$.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# local helper module, all repeated code lives in lungseg.py
import lungseg

# create a folder for figures if it doesn't exist
os.makedirs('figures', exist_ok=True)

from skimage import filters

In [ ]:
# pick one normal case to work with
name      = lungseg.list_cases(label=0)[0]
image, gt = lungseg.load_case(name)

print(f'Working on: {name}')

#### (4) Supervised thresholding

To choose a sensible value, I first plot the histogram of pixel intensities, split by the ground
truth so I can see where lung pixels actually sit.

In [ ]:
# naive threshold: the mean intensity, and Otsu's threshold for comparison
t_mean = image.mean()
t_otsu = filters.threshold_otsu(image)

# 100 equal-width brightness bins between 0 and 1
bins = np.linspace(0, 1, 101)

plt.figure(figsize=(8, 4))
plt.hist(image[~gt], bins=bins, alpha=0.6, label='not lung (ground truth)')
plt.hist(image[gt], bins=bins, alpha=0.6, label='lung (ground truth)')
plt.axvline(t_mean, color='k', ls='--', label=f'mean = {t_mean:.2f}')
plt.axvline(t_otsu, color='r', ls='-', label=f'Otsu = {t_otsu:.2f}')
plt.xlabel('Pixel intensity (0 = black, 1 = white)')
plt.ylabel('Number of pixels')
plt.legend()
plt.savefig('figures/fig_2_histogram.pdf', bbox_inches='tight')
plt.show()

The simplest (naive) choice is the mean intensity:

In [ ]:
# True wherever the pixel is darker than the mean
mask_mean = image < t_mean

print(f'Dice vs ground truth: {lungseg.dice(mask_mean, gt):.3f}')

The **Dice** coefficient measures overlap between two masks (1 = identical, 0 = no overlap) and is
discussed properly in Part 3.

Because the radiologist's mask is available, I can also test every threshold and pick the best one.
This is *supervised*, since it uses the ground truth, which would not be available for a new patient.

In [ ]:
# Dice score for 91 candidate thresholds
thresholds = np.linspace(0.05, 0.95, 91)
scores     = [lungseg.dice(image < t, gt) for t in thresholds]

# threshold with the highest Dice
t_best = thresholds[np.argmax(scores)]

plt.figure(figsize=(7, 4))
plt.plot(thresholds, scores)
plt.axvline(t_mean, color='k', ls='--', label='mean')
plt.axvline(t_otsu, color='r', label='Otsu')
plt.axvline(t_best, color='g', ls=':', label=f'best = {t_best:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Dice vs ground truth')
plt.legend()
plt.savefig('figures/fig_2_dice_vs_threshold.pdf', bbox_inches='tight')
plt.show()

#### (5) Unsupervised thresholding (Otsu's method)

Otsu's method tries every possible threshold and picks the one that makes the two groups each as
uniform as possible, i.e. it maximises the *between-class variance*. It needs no ground truth, so it
can be used on new patients.

In [ ]:
mask_otsu = image < t_otsu

masks  = [gt, mask_mean, mask_otsu, image < t_best]
titles = ['Ground truth', 'Mean threshold', 'Otsu threshold', 'Best (supervised)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, m, title in zip(axes, masks, titles):

    ax.imshow(m, cmap='gray')

    # add the Dice score to every mask except the ground truth itself
    if m is not gt:
        title += f'\nDice = {lungseg.dice(m, gt):.3f}'

    ax.set_title(title)
    ax.axis('off')

plt.savefig('figures/fig_2_threshold_masks.pdf', bbox_inches='tight')
plt.show()

Points for the write-up:

- All thresholds also select the **air outside the body** and leave holes and specks, which limits Dice.
- How close is Otsu to the best supervised threshold, and why might they differ?
- Thresholding uses **only brightness**, not shape or position. Part 3 addresses this with filtering and morphology.

Optional: repeat this notebook with a TB case by changing `label=0` to `label=1` in the third cell.